# AI Model Validation: From Mathematics to Trust
## Episode 17 — LLM Evaluation: How Do We Scientifically Measure Generative AI?

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

Episodes 1–16 built the foundations of measurement, uncertainty, generalization, robustness, reliability, safety, fairness, explainability and reproducibility.

Now the series enters generative AI.

> **How do we scientifically evaluate a system when many different answers can be valid?**

Traditional prediction often compares one output against one target. Generative AI produces open-ended language, code, plans, summaries, tool actions and reasoning artifacts.

### One Benchmark Score ≠ LLM Trust

Central proposition:

> **LLM evaluation is not the search for one perfect metric. It is the construction of evidence that a generative system satisfies the requirements of its intended use.**

Scientific chain:

### Task → Requirement → Test Case → Generation → Measurement → Judge → Uncertainty → Failure Analysis → Evidence → Trust Decision → Continuous Evaluation


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re, math, platform, sys
from collections import Counter
rng=np.random.default_rng(1717)
print("Python:",sys.version.split()[0])
print("NumPy:",np.__version__)
print("System:",platform.system(),platform.machine())


## 1. Why LLM Evaluation Is Different

For classification:

### ŷ ∈ {1,...,K}

For open-ended generation:

### y = (t₁,t₂,...,t_L)

There can be many acceptable sequences.

Therefore exact equality between generated text and one reference can be an inadequate definition of quality.


## 2. The Evaluation Object

Before choosing a metric, define what is being evaluated:

### Base Model
### Instruction-Tuned Model
### Prompt + Model
### RAG Pipeline
### Agent
### Tool-Using Workflow
### End-to-End Application

### Model Evaluation ≠ System Evaluation


## 3. Intended Use Comes First

There is no meaningful metric without a meaningful question.

For each use case define:

### User
### Task
### Context
### Expected Behaviour
### Failure Conditions
### Consequence
### Acceptance Criteria

Metrics should follow requirements—not the other way around.


## 4. Unit of Evaluation

Possible units include:

### Prompt
### Response
### Claim
### Sentence
### Conversation
### Tool Call
### Agent Trajectory
### Session
### User Outcome

The denominator determines the meaning of the metric.


## 5. Exact Match

For generated answer y and reference y*:

### EM=1[y=y*]

Exact match is useful when one canonical answer is required.

It is often too strict for semantically equivalent natural-language responses.


In [ ]:
reference="Paris"
answers=["Paris","paris","The answer is Paris.","France's capital is Paris.","Lyon"]
for a in answers:
    exact=int(a==reference)
    normalized=int(re.sub(r"[^a-z0-9 ]","",a.lower()).strip()==reference.lower())
    print(f"{a:28s} exact={exact} normalized_exact={normalized}")


## 6. Exact Match Failure Mode

Two answers can communicate the same fact using different surface forms.

### Lexical Difference ≠ Semantic Error

Conversely, high lexical overlap does not guarantee factual correctness.


## 7. Token Overlap

Let reference tokens form multiset R and candidate tokens C.

Precision-like overlap:

### P=Overlap/|C|

Recall-like overlap:

### R=Overlap/|R|

and:

### F1=2PR/(P+R)

Token overlap is transparent but remains lexical.


In [ ]:
def tokens(s):
    return re.findall(r"[a-z0-9]+",s.lower())
def token_f1(ref,cand):
    r=Counter(tokens(ref)); c=Counter(tokens(cand))
    overlap=sum((r & c).values())
    p=overlap/max(1,sum(c.values()))
    rec=overlap/max(1,sum(r.values()))
    f=0 if p+rec==0 else 2*p*rec/(p+rec)
    return p,rec,f

ref="The capital of France is Paris"
for cand in ["Paris is the capital of France",
             "France has Paris as its capital",
             "The capital of France is Lyon"]:
    print(cand,token_f1(ref,cand))


## 8. Lexical Metrics Have Limits

BLEU-, ROUGE- and token-overlap-style metrics can be useful for specific tasks and historical comparability.

But:

### Lexical Similarity ≠ Factuality
### Lexical Similarity ≠ Helpfulness
### Lexical Similarity ≠ Safety

Metric validity depends on the task.


## 9. Semantic Similarity

Represent texts as vectors:

### e_r = Embed(reference)
### e_c = Embed(candidate)

Cosine similarity:

### cos(e_r,e_c)=e_rᵀe_c/(||e_r||||e_c||)

Semantic similarity can tolerate paraphrases better than exact match.

It still does not prove factual correctness.


## 10. Similarity Can Reward the Wrong Answer

Consider:

**Reference:** Paris is the capital of France.

**Candidate:** Lyon is the capital of France.

The sentences are semantically and lexically very similar while one critical entity is wrong.

### High Semantic Similarity ≠ Correct Answer


## 11. Evaluation Must Be Decomposed

Instead of one vague “quality” score, evaluate dimensions such as:

### Correctness
### Relevance
### Completeness
### Groundedness
### Factuality
### Instruction Following
### Safety
### Style
### Consistency
### Latency
### Cost

Not every application needs every dimension.


## 12. Correctness

Correctness asks whether the answer satisfies task truth conditions.

For closed-form tasks this can be deterministic.

For open-ended tasks it may require references, executable tests, expert review, or structured rubrics.

### Correctness Metric Must Match the Task


## 13. Factuality

Factuality concerns whether factual claims are supported by reality or the authoritative source appropriate to the task.

A response can be fluent, relevant and stylistically excellent while containing false claims.

### Fluency ≠ Factuality


## 14. Claim-Level Evaluation

Decompose a response into claims:

### Response → {c₁,c₂,...,c_m}

For each claim classify, for example:

### Supported
### Contradicted
### Unverifiable / Insufficient Evidence

Then aggregate carefully.

Claim extraction itself can introduce measurement error.


In [ ]:
claims=["Claim 1","Claim 2","Claim 3","Claim 4","Claim 5"]
status=["supported","supported","contradicted","supported","unverifiable"]
supported=sum(s=="supported" for s in status)
contradicted=sum(s=="contradicted" for s in status)
print("Supported fraction:",supported/len(status))
print("Contradicted fraction:",contradicted/len(status))


## 15. Hallucination

“Hallucination” is used inconsistently.

A defensible evaluation should define the failure operationally, such as:

### Unsupported factual claim
### Contradiction with provided evidence
### Fabricated citation
### Invented entity
### Unjustified certainty

### Hallucination Requires a Testable Definition


## 16. Groundedness

For source context D and generated response Y:

### Groundedness asks whether claims in Y are supported by D.

This is especially important for RAG.

Groundedness is different from world factuality:

### Supported by Retrieved Context ≠ Necessarily True in the World


## 17. Groundedness Precision

If m claims are extracted and s are supported by the supplied context:

### Groundedness Precision = s/m

But this can reward an incomplete response that makes very few safe claims.

Therefore groundedness should be combined with completeness or requirement coverage.


## 18. Citation Correctness

For systems producing citations, evaluate:

### Citation Exists
### Citation Entails Claim
### Citation Is Relevant
### Citation Is Correctly Attached
### Citation Source Is Appropriate

A citation-shaped string is not evidence.

### Citation Presence ≠ Citation Support


## 19. Retrieval Evaluation

RAG quality depends on retrieval and generation.

For relevant set R and retrieved set K:

### Recall@k = |R∩K|/|R|

### Precision@k = |R∩K|/k

Generation cannot reliably use evidence that retrieval never supplied.


In [ ]:
relevant={"d2","d5","d7","d9"}
retrieved=["d5","d1","d7","d3","d8"]
hit=len(relevant.intersection(retrieved))
print("Recall@5:",hit/len(relevant))
print("Precision@5:",hit/len(retrieved))


## 20. Retrieval and Generation Must Be Separated

A wrong RAG answer can arise from:

### Retrieval Failure
### Ranking Failure
### Context Truncation
### Generation Failure
### Citation Failure

### End-to-End Failure ≠ Automatically Model Failure


## 21. Context Utilization

Even when correct evidence is retrieved, the generator may ignore it.

A useful test asks whether output quality changes appropriately when relevant evidence is added, removed, or contradicted.

This connects evaluation with perturbation testing.


## 22. Faithfulness to Context

A model can answer from memorized knowledge rather than the supplied context.

Depending on the application this can be acceptable—or a serious failure.

Therefore define whether the system is expected to:

### Use Only Context
### Prefer Context
### Combine Context and Model Knowledge
### Abstain When Context Is Insufficient


## 23. Completeness

Suppose a task has required elements:

### R={r₁,...,r_m}

and output covers subset C.

A simple requirement coverage measure is:

### Completeness=|C|/|R|

The requirements must themselves be valid and sufficiently granular.


## 24. Instruction Following

Convert an instruction into atomic constraints:

### C₁,C₂,...,C_m

Then evaluate:

### IF = (1/m)Σ_j1[C_j satisfied]

This is more auditable than asking whether a response “generally followed instructions.”


In [ ]:
constraints={
    "under_100_words":True,
    "contains_summary":True,
    "no_markdown_table":False,
    "mentions_limitations":True
}
score=np.mean(list(constraints.values()))
print("Instruction-following score:",score)
print(constraints)


## 25. Rubric Evaluation

A rubric maps qualitative requirements to explicit scoring anchors.

Example:

### 0 = Incorrect / unusable
### 1 = Major errors
### 2 = Partially correct
### 3 = Correct with minor omissions
### 4 = Fully correct and complete

The meaning of each level should be observable and task-specific.


## 26. Ordinal Scores Are Not Automatically Interval Scales

A rubric score of 4 is not necessarily “twice as good” as 2.

### Ordinal Rank ≠ Equal Numerical Distance

Averages of rubric scores can be operationally useful, but their interpretation should remain cautious.


## 27. Human Evaluation

Humans can evaluate nuanced criteria unavailable to deterministic metrics.

But human judgment introduces:

### Rater Variance
### Expertise Differences
### Fatigue
### Ambiguous Rubrics
### Preference Bias
### Order Effects

Human evaluation is a measurement process that must itself be validated.


## 28. Inter-Rater Agreement

If two raters classify outputs, Episode 16's agreement logic applies.

### κ=(P_o−P_e)/(1−P_e)

Low agreement may indicate ambiguous criteria, difficult examples, inadequate training, or genuine subjectivity.


## 29. LLM-as-a-Judge

An LLM can evaluate another model's output against a rubric.

Potential benefits:

### Scale
### Speed
### Repeatability
### Structured Feedback

Potential risks:

### Judge Bias
### Position Bias
### Self-Preference
### Prompt Sensitivity
### Version Drift
### Hallucinated Critiques

### LLM Judge ≠ Ground Truth


## 30. Judge Calibration

Compare automated judge decisions with trusted human labels.

For binary accept/reject judgment, compute:

### Accuracy
### Precision
### Recall
### Specificity
### Calibration if probabilistic

The evaluator should be evaluated before it is used as evidence.


In [ ]:
N=1000
human=rng.binomial(1,.65,N)
judge=np.where(rng.random(N)<.88,human,1-human)
tp=np.sum((human==1)&(judge==1))
tn=np.sum((human==0)&(judge==0))
fp=np.sum((human==0)&(judge==1))
fn=np.sum((human==1)&(judge==0))
print("Judge accuracy:",(tp+tn)/N)
print("Judge precision:",tp/(tp+fp))
print("Judge recall:",tp/(tp+fn))
print("Judge specificity:",tn/(tn+fp))


## 31. Position Bias

In pairwise evaluation, a judge may prefer the first or second response because of presentation order.

Test both:

### Judge(A,B)

and:

### Judge(B,A)

A robust preference should not depend strongly on position.


## 32. Pairwise Preference

Instead of absolute scoring, ask which response better satisfies a criterion.

For models A and B:

### p_AB=P(A preferred over B)

Estimate preference over many matched test cases.

Pairwise evaluation can be easier than absolute scoring, but remains judge-dependent.


In [ ]:
N=500
wins_A=287
p=wins_A/N
se=np.sqrt(p*(1-p)/N)
print("A preference rate:",p)
print("Approx 95% interval:",(p-1.96*se,p+1.96*se))


## 33. Ties Matter

For pairwise evaluation preserve:

### A Wins
### B Wins
### Tie

Forcing every ambiguous case into A or B can exaggerate differences.

### Forced Preference ≠ Evidence of Meaningful Difference


## 34. Pairwise Transitivity Is Not Guaranteed

Preferences can be cyclic:

### A > B
### B > C
### C > A

This can occur when systems have different strengths across tasks.

A single total ranking can hide multidimensional performance.


## 35. Elo-Style Ratings

Pairwise outcomes can be summarized through rating systems.

A logistic expected score can be written:

### P(A beats B)=1/[1+10^((R_B−R_A)/400)]

Ratings are relative to the comparison pool and evaluation protocol.

### Leaderboard Rating ≠ Absolute Capability


## 36. Pass@k

For tasks where multiple generated attempts are allowed:

### pass@k = P(at least one of k attempts succeeds)

Under independent attempts with per-attempt success p:

### pass@k=1−(1−p)^k

Real attempts may be correlated.


In [ ]:
for p in [.1,.3,.5,.8]:
    vals=[1-(1-p)**k for k in [1,2,5,10]]
    print("p=",p,"pass@1,2,5,10=",np.round(vals,4))


## 37. Unbiased pass@k Estimator for Sampled Candidates

If n candidates are sampled and c are correct, a common estimator for k≤n is:

### pass@k = 1 − C(n−c,k)/C(n,k)

when the combinatorial assumptions apply.

This differs from simply raising an empirical p under independence.


In [ ]:
def pass_at_k(n,c,k):
    if n-c < k:
        return 1.0
    return 1-math.comb(n-c,k)/math.comb(n,k)
for k in [1,2,5,10]:
    print("k=",k,"estimate=",pass_at_k(20,6,k))


## 38. Code Evaluation

For generated code, natural-language similarity is usually weak evidence.

Better evidence may include:

### Compilation
### Unit Tests
### Property Tests
### Security Tests
### Performance Tests
### Hidden Tests

### Similar Code Text ≠ Correct Program


## 39. Executable Evaluation

When outputs have executable semantics, evaluate behaviour directly.

Examples:

### SQL → execute query safely
### Code → run tests
### Structured JSON → validate schema
### Mathematics → verify result
### Tool plan → simulate or execute under controls

Executable tests often provide stronger evidence than text similarity.


## 40. Structured Output Validation

For JSON or schema-constrained output:

### Syntax Validity
### Schema Validity
### Required Fields
### Type Correctness
### Semantic Constraints

These can often be evaluated deterministically.


## 41. Conversation Evaluation

Single-turn quality can miss failures that emerge over a dialogue.

Evaluate:

### Context Retention
### Contradiction
### Goal Completion
### Recovery
### User Corrections
### Long-Horizon Consistency

### Good Turn ≠ Good Conversation


## 42. Agent Evaluation

For an agent, final-answer quality is only one dimension.

Measure:

### Goal Success
### Tool Correctness
### Step Efficiency
### Policy Compliance
### Recovery
### Cost
### Latency
### Safety
### Reproducibility

### Correct Final Answer ≠ Correct Agent Process


## 43. Trajectory Evaluation

Represent an agent trajectory:

### τ=(s₀,a₀,s₁,a₁,...,s_T)

Evaluate both outcome and path.

A system can reach the correct result through unsafe, wasteful, unauthorized or irreproducible actions.


## 44. Cost and Latency

Quality is often constrained by operational resources.

For task i:

### Cost_i
### Latency_i
### Quality_i

A useful system must satisfy a feasible operating region rather than maximize quality without constraint.


## 45. Cost Efficiency

One possible application-specific quantity is:

### Efficiency = Quality / Cost

But ratios can behave badly and can obscure minimum quality requirements.

A better decision may use constraints:

### maximize Quality subject to Cost≤C_max and Latency≤L_max


## 46. Multi-Metric Evaluation

Represent evidence as vector:

### M=(Correctness, Groundedness, Safety, Consistency, Latency, Cost,...)

Collapsing this immediately into one scalar can hide failure.

### Metric Vector → Evidence Profile

A weighted score is justified only when the decision semantics of the weights are explicit.


## 47. Metric Correlation

If several metrics are highly correlated, they may provide redundant evidence.

If they disagree, the disagreement can reveal different failure dimensions.

### More Metrics ≠ More Independent Evidence


## 48. Composite Scores

A weighted composite:

### S=Σ_jw_jM_j

requires choices about:

### Normalization
### Direction
### Scale
### Weights
### Missing Metrics
### Compensability

A severe safety failure should not necessarily be “compensated” by high style quality.


## 49. Hard Gates Before Composite Scores

A stronger architecture can use:

### Safety Gate
### Factuality Gate
### Reliability Gate

followed by optimization among systems that pass.

### Constraint Satisfaction → Then Ranking

This prevents unacceptable failure from disappearing inside an average.


## 50. Benchmark Design

A benchmark should define:

### Target Capability
### Task Distribution
### Sampling
### Difficulty
### Ground Truth
### Scoring
### Repetitions
### Uncertainty
### Leakage Controls
### Version

A benchmark is an experiment—not merely a spreadsheet of questions.


## 51. Benchmark Representativeness

Let:

### P_bench(X) = benchmark task distribution

### P_deploy(X) = deployment task distribution

If these differ materially:

### Benchmark Performance ≠ Deployment Performance

This is Episode 11 distribution shift applied to evaluation itself.


## 52. Benchmark Saturation

When models approach the ceiling of a benchmark, differences become harder to interpret.

High scores can indicate:

### Capability
### Easy Benchmark
### Contamination
### Narrow Coverage
### Ceiling Effects

Benchmark difficulty must evolve with systems.


## 53. Benchmark Contamination

If benchmark items or close variants appear in training data:

### Evaluation Independence Is Violated

Observed performance can overestimate generalization.

Contamination is difficult to rule out completely for large web-trained models, so claims should be calibrated accordingly.


## 54. Memorization vs Generalization

A model answering a known benchmark item correctly may be using learned general capability—or memorized exposure.

Fresh, private, procedurally generated, or time-separated tests can reduce some contamination risks.

### Correct Benchmark Answer ≠ Proven Generalization


## 55. Dynamic Evaluation

Static benchmarks become stale.

Dynamic evaluation can rotate:

### Fresh Questions
### New Documents
### Current Tasks
### Adversarial Cases
### Real Failure Cases

Continuous evaluation should preserve comparability while introducing fresh evidence.


## 56. Challenge Sets

Average random samples may underrepresent important failures.

Create targeted challenge sets for:

### Negation
### Ambiguity
### Long Context
### Rare Entities
### Conflicting Evidence
### Prompt Injection
### Multilingual Inputs
### Tool Failures
### Edge Cases

Challenge sets complement representative sampling.


## 57. Representative Set vs Challenge Set

Representative sets estimate expected deployment performance.

Challenge sets probe known vulnerabilities.

### Representative Evidence ≠ Stress Evidence

Both are needed for different scientific questions.


## 58. Difficulty Stratification

Report performance by task difficulty:

### M_easy
### M_medium
### M_hard

An aggregate can hide collapse on difficult tasks.

Difficulty labels themselves should be independently justified where possible.


## 59. Slice Evaluation

Evaluate conditional performance:

### M | Domain
### M | Language
### M | Task Type
### M | Context Length
### M | User Group
### M | Risk Level

This connects LLM evaluation to subgroup validation.


## 60. Long-Context Evaluation

A model's advertised context window does not establish effective use of information across that window.

Test:

### Retrieval Position
### Distractor Density
### Multi-Hop Dependency
### Contradiction
### Recency
### Context Utilization

### Context Capacity ≠ Context Competence


## 61. Robustness of LLM Evaluation

Perturb prompts while preserving intended meaning.

Measure:

### ΔM = M(original)−M(perturbed)

Robust LLM evaluation should include paraphrases, formatting changes, irrelevant context and realistic noise.


## 62. Consistency Across Repeated Generations

Episode 16 applies directly.

For task i and repeated generation r:

### M_ir

Measure:

### Mean
### Variance
### Failure Probability
### Lower Quantiles

### One Good Completion ≠ Stable Capability


## 63. Best-of-N Can Hide Instability

If only the best output among N attempts is reported, performance can appear much stronger than a typical user experience.

Always distinguish:

### pass@1
### pass@k
### best-of-N
### average-of-N

These are different operational claims.


## 64. Self-Consistency

Generate multiple candidate reasoning paths or answers and aggregate them.

If each independent candidate is correct with probability p, majority-vote accuracy can be computed under simplified independence assumptions.

Real generations are correlated, so theoretical gains can be overstated.


In [ ]:
def majority_prob(n,p):
    start=n//2+1
    return sum(math.comb(n,k)*(p**k)*((1-p)**(n-k)) for k in range(start,n+1))
for p in [.55,.65,.75]:
    print("p=",p,"majority n=5:",majority_prob(5,p),"n=11:",majority_prob(11,p))


## 65. Correlated Samples

Repeated LLM outputs share the same model, prompt and learned representations.

Therefore errors can be strongly correlated.

### k Samples ≠ k Independent Opinions

Self-consistency should be empirically validated rather than assumed.


## 66. Confidence Is Difficult in Generative AI

Token probabilities are not automatically calibrated probabilities that an entire answer is correct.

Sequence likelihood can prefer fluent common text rather than factual truth.

### High Token Probability ≠ High Answer Correctness


## 67. Abstention

For uncertain tasks, an LLM may abstain or escalate.

Evaluate:

### Coverage=P(Model Answers)

### Selective Risk=E[L|Model Answers]

A useful model should know when the evidence is insufficient—but abstention quality itself requires validation.


## 68. Uncertainty Through Repetition

Disagreement across repeated outputs can be an uncertainty signal.

But:

### Agreement ≠ Correctness
### Disagreement ≠ Incorrectness

Use repeated-generation uncertainty as one evidence source, not a truth detector.


## 69. Evaluation Uncertainty

Every reported LLM metric is estimated from finite test cases and possibly stochastic generations.

Preserve:

### Sample Size
### Standard Error
### Confidence Interval
### Repetition Count
### Judge Variance

### Score Without Uncertainty = Incomplete Evidence


## 70. Bootstrap Confidence Intervals

For task-level scores M₁,...,M_n, bootstrap tasks to estimate uncertainty in the aggregate.

This is useful when analytic standard errors are inconvenient, provided the resampling unit matches the experimental design.


In [ ]:
n=300
task_scores=np.clip(rng.beta(8,2,n),0,1)
B=3000
means=np.empty(B)
for b in range(B):
    means[b]=rng.choice(task_scores,size=n,replace=True).mean()
print("Observed mean:",task_scores.mean())
print("Bootstrap 95% CI:",np.quantile(means,[.025,.975]))


## 71. Paired Model Comparison

When models A and B answer the same tasks, compare paired differences:

### D_i=M_A,i−M_B,i

This removes some task-difficulty variation.

Report:

### D̄
### Confidence Interval
### Win/Tie/Loss
### Failure-Type Differences

### Higher Mean Alone ≠ Complete Comparison


## 72. Practical Significance

A statistically detectable 0.3-point improvement may not matter operationally.

A small average improvement may still matter if it eliminates a severe failure mode.

### Statistical Significance ≠ Practical Significance


## 73. Multiple Model Selection

If many models, prompts and configurations are tested and only the best is reported, selection bias occurs.

### Winner's Curse

The selected configuration should be confirmed on untouched evidence.


## 74. Evaluation Overfitting

Repeated optimization against the same benchmark can overfit the benchmark even without conventional training.

### Benchmark Feedback → Configuration Changes → Benchmark Reuse

Eventually the benchmark becomes part of the development process rather than independent validation.


## 75. Development vs Validation Sets

Maintain separation:

### Development Evaluation
### Validation Evaluation
### Final / External Evaluation

The more a test set influences decisions, the less independent it becomes.


## 76. Failure Taxonomy

Do not store only scores.

Classify failures such as:

### Factual Error
### Unsupported Claim
### Missing Requirement
### Reasoning Error
### Retrieval Error
### Tool Error
### Safety Violation
### Formatting Failure
### Refusal Error
### Overconfidence

Failure categories support engineering remediation.


## 77. Severity-Weighted Failure Analysis

Not all errors have equal consequence.

For failure class j:

### Risk_j=P(F_j)C_j

Episode 13 safety mathematics therefore belongs inside LLM evaluation when consequences differ materially.


## 78. Error Clustering

A model may fail repeatedly on the same semantic pattern.

Cluster failures by:

### Task
### Domain
### Cause
### Prompt Pattern
### Tool
### Language
### Severity

This reveals systematic weaknesses hidden by aggregate rates.


## 79. Red-Team Evaluation

Adversarial evaluation intentionally searches for failures.

It complements representative benchmarks.

Measure not only whether a jailbreak or failure occurs, but:

### Attack Class
### Attempts
### Success Rate
### Severity
### Transferability
### Reproducibility
### Defense Effectiveness


## 80. Safety Evaluation

Safety cannot be reduced to refusal rate.

Evaluate both:

### Unsafe Compliance
### Excessive Refusal

A model that refuses everything may appear safe under one metric while being useless.

### Safety ≠ Maximum Refusal


## 81. Over-Refusal

For benign request B:

### OverRefusal=P(Refuse|B)

For harmful request H:

### UnsafeCompliance=P(Comply|H)

A useful safety evaluation studies both error directions.


## 82. Multi-Turn Safety

A model safe on isolated prompts can fail after:

### Context Accumulation
### Role Changes
### Indirect Instructions
### Tool Feedback
### Long-Horizon Planning

Safety evaluation must match the interaction horizon.


## 83. Evaluation of Tool Use

For tool calls evaluate:

### Correct Tool
### Correct Arguments
### Authorization
### Sequence
### Result Interpretation
### Recovery on Failure

A linguistically excellent answer does not compensate for an unsafe tool action.


## 84. Agent Goal Completion

Define goal success at the task level:

### Success=1[Required terminal conditions satisfied]

Then separately evaluate trajectory efficiency and safety.

### Goal Success ≠ Process Quality


## 85. Agent Efficiency

For successful runs:

### Steps
### Tokens
### Tool Calls
### Cost
### Latency

Efficiency should usually be evaluated subject to correctness and safety rather than as a substitute for them.


## 86. Evaluation Data Lineage

Every test case should preserve:

### Source
### Author
### Timestamp
### License/Permission where relevant
### Version
### Transformation
### Expected Behaviour
### Risk Category

Evaluation data is part of the evidence chain.


## 87. Evaluation Versioning

Version:

### Dataset
### Rubric
### Judge
### Prompt
### Metric Implementation
### Model
### Retrieval Corpus
### Tool Environment

Otherwise score changes cannot be scientifically attributed.


## 88. Evidence Record

For every evaluation result preserve:

### Claim
### Requirement
### Test Set
### Sampling Design
### Model/System Version
### Configuration
### Metric
### Judge
### Raw Outcomes
### Aggregate
### Uncertainty
### Failures
### Limitations
### Decision

### Metric Without Provenance ≠ Reproducible Evidence


## 89. OpenVals Evaluation Architecture

A rigorous OpenVals evaluation layer can separate:

### Dataset
### Runner
### Model Adapter
### Metric
### Judge
### Failure Taxonomy
### Evidence Store
### Benchmark Comparison
### Continuous Monitor

This allows new models, judges and metrics without destroying evidence lineage.


## 90. OpenVals Multi-Dimensional Evidence

OpenVals should preserve dimensions rather than prematurely collapsing them:

### Accuracy / Correctness
### Semantic Similarity
### Factuality
### Groundedness
### Reliability
### Safety
### Consistency
### Variance
### Latency
### Cost Efficiency
### Robustness
### Trust Evidence

A trust decision should remain traceable to underlying measurements.


## 91. The Danger of the Universal LLM Score

A single score can hide:

### Catastrophic Tail Failure
### Poor Groundedness
### Subgroup Failure
### High Cost
### Instability
### Unsafe Tool Use

### Scalar Ranking ≠ Complete Assurance

Use scalar summaries only when the decision model justifies them.


## 92. Evaluation as Measurement Science

A scientific LLM evaluation asks:

### What construct are we measuring?
### Is the metric valid for that construct?
### Is the measurement reliable?
### What is the uncertainty?
### Does the benchmark represent deployment?
### Can the result be reproduced?
### What failures are hidden by the average?

This turns evaluation into measurement science.


## 93. Evaluation Ladder

A useful progression is:

### Exact Match
### Task Metric
### Semantic Metric
### Rubric
### Human / Judge Evaluation
### Failure Analysis
### Stress Testing
### Repeated Evaluation
### Continuous Validation
### Assurance

No single rung replaces the others.


## 94. Scientific LLM Evaluation Chain

### Intended Use → Requirement → Dataset → Generation → Metric/Judge → Uncertainty → Failure Analysis → Stress Test → Evidence → Decision → Monitoring → Revalidation

This is the bridge from benchmark culture to AI assurance.


## 95. Mathematical Summary

### EM=1[y=y*]

### F1=2PR/(P+R)

### cos(a,b)=aᵀb/(||a||||b||)

### Groundedness=supported claims/claims

### Recall@k=|R∩K|/|R|

### Precision@k=|R∩K|/k

### Completeness=covered requirements/required requirements

### κ=(P_o−P_e)/(1−P_e)

### p_AB=P(A preferred over B)

### pass@k=1−(1−p)^k  under independent attempts

### pass@k=1−C(n−c,k)/C(n,k)  for the sampled-candidate estimator

### M=(M₁,M₂,...,M_d)

### S=Σ_jw_jM_j  only when aggregation semantics are justified

### D_i=M_A,i−M_B,i

### Risk_j=P(F_j)C_j

### Coverage=P(Model Answers)

### Selective Risk=E[L|Model Answers]


## 96. Central Conclusion

### Exact Match ≠ Semantic Quality
### Semantic Similarity ≠ Factuality
### Groundedness ≠ World Truth
### LLM Judge ≠ Ground Truth
### Benchmark Score ≠ Deployment Performance
### Best-of-N ≠ Typical User Experience
### More Metrics ≠ More Independent Evidence
### One Score ≠ LLM Trust

> **LLM evaluation is not the search for one perfect metric. It is the construction of evidence that a generative system satisfies the requirements of its intended use.**

The scientific questions are:

### What exactly are we evaluating?
### What behaviour is required?
### What constitutes failure?
### Which measurements validly represent those requirements?
### Who or what judges the output?
### How reliable is that judge?
### What uncertainty surrounds the score?
### Does the benchmark represent deployment?
### Is the benchmark contaminated?
### How does the system behave under repetition and stress?
### Which failures are hidden by the average?
### Does the evidence remain valid after the model, prompt, corpus or environment changes?

OpenVals principle:

> **A metric is a number. LLM trust requires an evidence system.**

## Next Episode

**Episode 18 — LLM Hallucination, Factuality, and Groundedness: How Do We Measure Whether Generative AI Is Telling the Truth?**

Episode 18 can go deeper into atomic claim extraction, entailment, contradiction, source attribution, citation validation, retrieval-grounded generation, unsupported-claim rates, factual precision/recall, uncertainty, abstention, knowledge boundaries and hallucination under distribution shift.
